In [ ]:
from pathlib import Path

from plant_shape_analysis import LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_leaf_sequence

%load_ext autoreload
%autoreload 2

In [ ]:
selected_sequences = [
"maize_control_plant1", # OK
"maize_control_plant2", # OK
"maize_control_plant3", # OK
"sorghum_control_plant1", # OK
"sorghum_control_plant2", # OK
"sorghum_highlight_plant2", # OK
"tobacco_control_plant1", # OK, FIXED alignment issue (last timestep)
"tobacco_control_plant2", # OK
"tobacco_shade_plant3", # OK, FIXED alignment issue (first 4 timesteps) and wrong labels
"tomato1_heat_plant3", # OK
"tomato1_shade_plant1", # OK
"tomato2_control_plant3" # OK
]

### Get data

In [ ]:
# Load dataset
dataset_path = Path("../data/TrackPlant3D/versions")
leaf_dataset = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    version="v2",
    apply_alignment=True,
    plant_alignment_method=None,
    estimate_plant_normals=False,
    auto_download=False,
    min_timepoints=1, # so we flip labels on single-timestep leaves too if needed for consistency
)

In [ ]:
# # Visualize all leaves of selected sequences
# selected_leaves = []
# for plant_name in sorted(selected_sequences):
#     all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
#     selected_leaves.extend([leaf["sequence_name"] for leaf in all_leaves])
# selected_leaves[:5], len(selected_leaves) # 55 leaf sequences to inspect

#### Workflow to flip normals for specific timepoints in a leaf sequence

We go plant by plant, visualize each leaf sequence, identify problematic timepoints, and flip normals as needed, then save the changes for the whole plants, and proceed to the next plant

In [ ]:
selected_leaves_by_plant = {}
for plant_name in sorted(selected_sequences):
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    selected_leaves_by_plant[plant_name] = [leaf["sequence_name"] for leaf in all_leaves]

In [ ]:
current_plant_idx = 5  # Change this to move between plants
current_plant = sorted(selected_sequences)[current_plant_idx]
selected_leaves = selected_leaves_by_plant[current_plant]
print(f"Number of leaves in {current_plant}: {len(selected_leaves)}")
selected_leaves

In [ ]:
# Visualize leaf to identify problematic timepoints
#leaf_name = "maize_control_plant2_leaf3"
leaf_name = selected_leaves[4]  # Change index to visualize different leaves
leaf_seq = leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
print(f"{leaf_name}, timesteps: {len(leaf_seq['timepoints'])}")

In [ ]:
# Visualize leaf sequence
visualize_leaf_sequence(leaf_seq, window_name=leaf_name, show_coordinate_frame=True)

In [ ]:
# Apply manual corrections
# List the timepoint INDICES (0-based) that need normal flipping
# Example: if timepoint 1 and 3 are wrong, use [1, 3]
timepoints_to_flip = [1]  # MODIFY THIS based on visual inspection
# timepoints_to_flip = list(range(len(leaf_seq["timepoints"]))) # Or uncomment this if we need to flip normals on all timepoints

# Apply the flip
leaf_dataset.flip_normals_for_timepoints(leaf_name, timepoints_to_flip)

# Visualize leaf again
visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_name}_corrected", show_coordinate_frame=True)

# RUN NEXT CELL ONLY WHEN READY WITH CHANGES IN CURRENT PLANT!

In [ ]:
output_dir = dataset_path / "v2"
leaf_dataset.save_plant_with_all_leaves(current_plant, output_dir=output_dir)

Approach below would also work for a single leaf, but better to go per plant, so we don't need to reload leaf dataset each time.

In [ ]:
# # Save corrected leaf back to plant PLY
# # This maps the corrected normals back to the plant and saves PLY files
# output_dir = dataset_path / "v2" / "gt_corrected_v2"
# leaf_dataset.save_leaf_to_plant_ply(leaf_name, output_dir=output_dir)

# # Must RELOAD to get updated normals (in leaf_dataset.plant_dataset and avoid disregarding previous changes when correcting a new leaf, since method uses plant_dataset internally which doesn't have the changes yet)
# leaf_dataset = LeafSequencesDataset(
#     dataset_path,
#     use_ply=True,
#     version="v2",
#     apply_alignment=True,
#     plant_alignment_method=None,
#     estimate_plant_normals=False,
#     auto_download=False,
#     min_timepoints=1, # so we flip labels on single-timestep leaves too if needed for consistency
# )